In [20]:
# # Notebook 3: Modelado y Predicción con K-means
# 
# Clustering K-means sobre 3 versiones de datos para identificar zonas de riesgo.
# Comparación de métricas y mapas entre versiones.

In [21]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'

OUT = 'output/'

## 1. Carga de datos - 3 versiones

In [22]:
gdf_v1 = gpd.read_file('output/zonas_ageb_clean.geojson')
gdf_v2 = gpd.read_file('output/zonas_ageb_clean_v2.json')
gdf_v3 = gpd.read_file('output/zonas_ageb_clean_v3.json')

print(f"v1: {gdf_v1.shape} - {gdf_v1.columns.tolist()}")
print(f"v2: {gdf_v2.shape}")
print(f"v3: {gdf_v3.shape}")

v1: (1018, 16) - ['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'tipo_suelo', 'area_total', 'riesgo_sismo', 'riesgo_inundacion', 'pct_afectacion_inundacion', 'suma_riesgos', 'riesgo_general', 'area_total_norm', 'pct_afectacion_inundacion_norm', 'tipo_suelo_label', 'geometry']
v2: (2453, 44)
v3: (2453, 42)


## 2. Preparación de features para clustering

Se seleccionan las variables numéricas de riesgo disponibles en cada versión.

In [23]:
def prepare_features(gdf, version_name):
    """Selecciona y escala features para K-means según la versión."""
    
    # Features comunes a todas las versiones
    base_features = ['riesgo_sismo', 'riesgo_inundacion', 'tipo_suelo', 'area_total']
    
    # Features adicionales por versión
    extra_features = {
        'v1': ['pct_afectacion_inundacion', 'suma_riesgos'],
        'v2': ['riesgo_laderas', 'pct_afectacion_sismo', 'pct_afectacion_inundacion',
               'pct_afectacion_laderas', 'severidad_inundacion', 'fracturas_count',
               'fracturas_longitud_m', 'ruse_emergencias', 'pob_total', 'imu_2020'],
        'v3': ['riesgo_laderas', 'pct_afectacion_sismo', 'pct_afectacion_inundacion',
               'pct_afectacion_laderas', 'severidad_inundacion', 'fracturas_count',
               'fracturas_longitud_m', 'ruse_emergencias', 'pob_total', 'imu_2020'],
    }
    
    features = base_features + extra_features.get(version_name, [])
    features = [f for f in features if f in gdf.columns]
    
    X = gdf[features].copy()
    X = X.fillna(0)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"\n{version_name}: {len(features)} features - {features}")
    return X_scaled, features

X_v1, feat_v1 = prepare_features(gdf_v1, 'v1')
X_v2, feat_v2 = prepare_features(gdf_v2, 'v2')
X_v3, feat_v3 = prepare_features(gdf_v3, 'v3')


v1: 6 features - ['riesgo_sismo', 'riesgo_inundacion', 'tipo_suelo', 'area_total', 'pct_afectacion_inundacion', 'suma_riesgos']

v2: 14 features - ['riesgo_sismo', 'riesgo_inundacion', 'tipo_suelo', 'area_total', 'riesgo_laderas', 'pct_afectacion_sismo', 'pct_afectacion_inundacion', 'pct_afectacion_laderas', 'severidad_inundacion', 'fracturas_count', 'fracturas_longitud_m', 'ruse_emergencias', 'pob_total', 'imu_2020']

v3: 14 features - ['riesgo_sismo', 'riesgo_inundacion', 'tipo_suelo', 'area_total', 'riesgo_laderas', 'pct_afectacion_sismo', 'pct_afectacion_inundacion', 'pct_afectacion_laderas', 'severidad_inundacion', 'fracturas_count', 'fracturas_longitud_m', 'ruse_emergencias', 'pob_total', 'imu_2020']


## 3. Elbow Method - Determinar k óptimo

In [24]:
def elbow_analysis(X, version_name, k_range=range(2, 11)):
    """Calcula inertia y silhouette para rango de k."""
    inertias = []
    silhouettes = []
    
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X)
        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(X, labels))
    
    return list(k_range), inertias, silhouettes

k_range = range(2, 11)
results = {}
for name, X in [('v1', X_v1), ('v2', X_v2), ('v3', X_v3)]:
    ks, inertias, sils = elbow_analysis(X, name, k_range)
    results[name] = {'ks': ks, 'inertias': inertias, 'silhouettes': sils}
    best_k = ks[np.argmax(sils)]
    print(f"{name}: Mejor silhouette en k={best_k} ({max(sils):.4f})")

v1: Mejor silhouette en k=2 (0.6652)
v2: Mejor silhouette en k=2 (0.3008)
v3: Mejor silhouette en k=2 (0.3681)


In [25]:
# Gráfica Elbow + Silhouette
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Selección de k - Elbow Method y Silhouette Score', fontsize=14, fontweight='bold')

colors = {'v1': '#3498db', 'v2': '#e74c3c', 'v3': '#2ecc71'}

for name, res in results.items():
    axes[0].plot(res['ks'], res['inertias'], 'o-', label=name, color=colors[name], linewidth=2)
    axes[1].plot(res['ks'], res['silhouettes'], 'o-', label=name, color=colors[name], linewidth=2)

axes[0].set_title('Elbow Method (Inercia)', fontsize=12)
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Silhouette Score', fontsize=12)
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT}charts/elbow_silhouette_comparativo.png', bbox_inches='tight')
plt.close()
print("✅ Guardado: elbow_silhouette_comparativo.png")

✅ Guardado: elbow_silhouette_comparativo.png


## 4. K-means con k=4 (clusters de riesgo) - Por cada versión

In [26]:
K_OPTIMAL = 4  # 4 niveles de riesgo natural

def run_kmeans(X, gdf, k, version_name):
    """Ejecuta K-means y asigna etiquetas ordenadas por riesgo."""
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=300)
    labels = km.fit_predict(X)
    
    # Métricas
    sil = silhouette_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    db = davies_bouldin_score(X, labels)
    
    # Ordenar clusters por riesgo medio (usando riesgo_sismo como proxy)
    gdf_temp = gdf.copy()
    gdf_temp['cluster_raw'] = labels
    
    # Calcular riesgo medio por cluster
    risk_col = 'riesgo_sismo' if 'riesgo_sismo' in gdf.columns else gdf.select_dtypes('number').columns[0]
    cluster_risk_mean = gdf_temp.groupby('cluster_raw')[risk_col].mean().sort_values()
    
    # Reasignar: 0=bajo riesgo, k-1=alto riesgo
    mapping = {old: new for new, old in enumerate(cluster_risk_mean.index)}
    gdf_temp['cluster'] = gdf_temp['cluster_raw'].map(mapping)
    
    metrics = {
        'version': version_name,
        'k': k,
        'silhouette': sil,
        'calinski_harabasz': ch,
        'davies_bouldin': db,
        'inertia': km.inertia_
    }
    
    print(f"\n=== {version_name} (k={k}) ===")
    print(f"  Silhouette Score: {sil:.4f}")
    print(f"  Calinski-Harabasz: {ch:.2f}")
    print(f"  Davies-Bouldin: {db:.4f}")
    print(f"  Distribución clusters: {gdf_temp['cluster'].value_counts().sort_index().to_dict()}")
    
    return gdf_temp, metrics

gdf_v1_km, metrics_v1 = run_kmeans(X_v1, gdf_v1, K_OPTIMAL, 'v1')
gdf_v2_km, metrics_v2 = run_kmeans(X_v2, gdf_v2, K_OPTIMAL, 'v2')
gdf_v3_km, metrics_v3 = run_kmeans(X_v3, gdf_v3, K_OPTIMAL, 'v3')


=== v1 (k=4) ===
  Silhouette Score: 0.4960
  Calinski-Harabasz: 1476.59
  Davies-Bouldin: 0.8509
  Distribución clusters: {0: 159, 1: 137, 2: 193, 3: 529}

=== v2 (k=4) ===
  Silhouette Score: 0.1694
  Calinski-Harabasz: 537.36
  Davies-Bouldin: 1.7751
  Distribución clusters: {0: 665, 1: 217, 2: 908, 3: 663}

=== v3 (k=4) ===
  Silhouette Score: 0.2812
  Calinski-Harabasz: 871.19
  Davies-Bouldin: 1.5699
  Distribución clusters: {0: 283, 1: 596, 2: 372, 3: 1202}


## 5. Mapas de clusters por versión

In [27]:
cluster_cmap = mcolors.ListedColormap(['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'])
cluster_labels = {0: 'Bajo', 1: 'Medio-Bajo', 2: 'Medio-Alto', 3: 'Alto'}

def plot_cluster_map(gdf, version_name, filename):
    """Genera mapa estático de clusters."""
    fig, ax = plt.subplots(figsize=(12, 12))
    gdf.plot(column='cluster', cmap=cluster_cmap, legend=False,
             ax=ax, edgecolor='gray', linewidth=0.2, alpha=0.85,
             vmin=0, vmax=3)
    ax.set_title(f'Clusters K-means (k={K_OPTIMAL}) - {version_name}', 
                 fontsize=14, fontweight='bold')
    ax.set_axis_off()
    
    # Agregar leyenda con labels
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, edgecolor='gray', label=l) 
                       for c, l in zip(['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'],
                                        ['Bajo', 'Medio-Bajo', 'Medio-Alto', 'Alto'])]
    ax.legend(handles=legend_elements, loc='lower left', fontsize=10, title='Cluster')
    
    plt.tight_layout()
    plt.savefig(f'{OUT}maps/{filename}', bbox_inches='tight')
    plt.close()
    print(f"✅ Guardado: maps/{filename}")

plot_cluster_map(gdf_v1_km, 'v1 (GeoJSON original)', 'mapa_clusters_v1.png')
plot_cluster_map(gdf_v2_km, 'v2 (JSON limpio)', 'mapa_clusters_v2.png')
plot_cluster_map(gdf_v3_km, 'v3 (Limpieza mejorada)', 'mapa_clusters_v3.png')

✅ Guardado: maps/mapa_clusters_v1.png
✅ Guardado: maps/mapa_clusters_v2.png
✅ Guardado: maps/mapa_clusters_v3.png


## 6. Análisis comparativo de las 3 versiones

In [28]:
# Tabla de métricas comparativa
metrics_df = pd.DataFrame([metrics_v1, metrics_v2, metrics_v3])
print("=== Métricas Comparativas de Clustering ===")
print(metrics_df.to_string(index=False))

=== Métricas Comparativas de Clustering ===
version  k  silhouette  calinski_harabasz  davies_bouldin      inertia
     v1  4    0.495978        1476.590136        0.850943  1137.724690
     v2  4    0.169391         537.358609        1.775066 20709.674377
     v3  4    0.281241         871.192737        1.569872 16612.791843


In [29]:
# Gráfica comparativa de métricas
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Comparación de Métricas de Clustering (k=4)', fontsize=14, fontweight='bold')

bar_colors = ['#3498db', '#e74c3c', '#2ecc71']
versions = ['v1', 'v2', 'v3']

# Silhouette (mayor = mejor)
vals = [metrics_v1['silhouette'], metrics_v2['silhouette'], metrics_v3['silhouette']]
bars = axes[0].bar(versions, vals, color=bar_colors, edgecolor='white')
axes[0].set_title('Silhouette Score\n(mayor = mejor)', fontsize=11)
axes[0].set_ylabel('Score')
for b, v in zip(bars, vals):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.005, f'{v:.4f}', 
                 ha='center', fontsize=10)

# Calinski-Harabasz (mayor = mejor)
vals = [metrics_v1['calinski_harabasz'], metrics_v2['calinski_harabasz'], metrics_v3['calinski_harabasz']]
bars = axes[1].bar(versions, vals, color=bar_colors, edgecolor='white')
axes[1].set_title('Calinski-Harabasz\n(mayor = mejor)', fontsize=11)
axes[1].set_ylabel('Score')
for b, v in zip(bars, vals):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+5, f'{v:.0f}', 
                 ha='center', fontsize=10)

# Davies-Bouldin (menor = mejor)
vals = [metrics_v1['davies_bouldin'], metrics_v2['davies_bouldin'], metrics_v3['davies_bouldin']]
bars = axes[2].bar(versions, vals, color=bar_colors, edgecolor='white')
axes[2].set_title('Davies-Bouldin\n(menor = mejor)', fontsize=11)
axes[2].set_ylabel('Score')
for b, v in zip(bars, vals):
    axes[2].text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{v:.4f}', 
                 ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(f'{OUT}charts/metricas_comparativas.png', bbox_inches='tight')
plt.close()
print("✅ Guardado: metricas_comparativas.png")

✅ Guardado: metricas_comparativas.png


In [30]:
# Mapas comparativos lado a lado
fig, axes = plt.subplots(1, 3, figsize=(24, 10))
fig.suptitle('Comparación de Clusters K-means - Todas las versiones', 
             fontsize=16, fontweight='bold')

for ax, gdf, name in [(axes[0], gdf_v1_km, 'v1 (1018 zonas, 6 features)'),
                       (axes[1], gdf_v2_km, 'v2 (2453 zonas, 14 features)'),
                       (axes[2], gdf_v3_km, 'v3 (2453 zonas, 14 features)')]:
    gdf.plot(column='cluster', cmap=cluster_cmap, ax=ax,
             edgecolor='gray', linewidth=0.15, alpha=0.85,
             vmin=0, vmax=3)
    ax.set_title(name, fontsize=12)
    ax.set_axis_off()
    
    # Stats
    n_alto = (gdf['cluster'] == 3).sum()
    pct_alto = n_alto / len(gdf) * 100
    ax.text(0.02, 0.02, f'Alto riesgo: {n_alto} ({pct_alto:.1f}%)',
            transform=ax.transAxes, fontsize=10, verticalalignment='bottom',
            bbox=dict(boxstyle='round', facecolor='#e74c3c', alpha=0.3))

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, edgecolor='gray', label=l) 
                   for c, l in zip(['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'],
                                    ['Bajo', 'Medio-Bajo', 'Medio-Alto', 'Alto'])]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=12,
           bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig(f'{OUT}maps/mapa_comparativo_clusters.png', bbox_inches='tight')
plt.close()
print("✅ Guardado: mapa_comparativo_clusters.png")

✅ Guardado: mapa_comparativo_clusters.png


## 7. Zonas de mayor riesgo por versión

In [31]:
print("=== ZONAS DE MAYOR RIESGO (Cluster Alto) ===\n")

for name, gdf in [('v1', gdf_v1_km), ('v2', gdf_v2_km), ('v3', gdf_v3_km)]:
    alto = gdf[gdf['cluster'] == 3]
    print(f"\n--- {name}: {len(alto)} zonas en cluster Alto ---")
    if 'CVE_MUN' in gdf.columns:
        print(f"  Alcaldías afectadas: {sorted(alto['CVE_MUN'].unique())}")
    
    # Estadísticas del cluster alto
    risk_cols = [c for c in ['riesgo_sismo', 'riesgo_inundacion', 'riesgo_laderas', 
                              'fracturas_count'] if c in alto.columns]
    if risk_cols:
        print(f"  Promedios cluster Alto:")
        for col in risk_cols:
            print(f"    {col}: {alto[col].mean():.2f}")

=== ZONAS DE MAYOR RIESGO (Cluster Alto) ===


--- v1: 529 zonas en cluster Alto ---
  Alcaldías afectadas: ['005', '007', '014', '015']
  Promedios cluster Alto:
    riesgo_sismo: 5.00
    riesgo_inundacion: 4.92

--- v2: 663 zonas en cluster Alto ---
  Alcaldías afectadas: ['002', '003', '005', '006', '007', '011', '012', '013', '014', '015', '016', '017']
  Promedios cluster Alto:
    riesgo_sismo: 4.62
    riesgo_inundacion: 4.86
    riesgo_laderas: 1.00
    fracturas_count: 2.30

--- v3: 1202 zonas en cluster Alto ---
  Alcaldías afectadas: ['002', '003', '005', '006', '007', '011', '012', '013', '014', '015', '016', '017']
  Promedios cluster Alto:
    riesgo_sismo: 4.57
    riesgo_inundacion: 4.87
    riesgo_laderas: 1.00
    fracturas_count: 0.04


In [32]:
# Top 15 zonas más riesgosas según v3
print("\n=== TOP 15 ZONAS DE MAYOR RIESGO (v3) ===")
if 'indice_riesgo_compuesto' in gdf_v3_km.columns:
    top15 = gdf_v3_km.nlargest(15, 'indice_riesgo_compuesto')
    show_cols = ['CVEGEO', 'CVE_MUN', 'cluster', 'riesgo_sismo', 'riesgo_inundacion', 
                 'riesgo_laderas', 'fracturas_count', 'indice_riesgo_compuesto']
    show_cols = [c for c in show_cols if c in top15.columns]
    print(top15[show_cols].to_string(index=False))


=== TOP 15 ZONAS DE MAYOR RIESGO (v3) ===
       CVEGEO CVE_MUN  cluster  riesgo_sismo  riesgo_inundacion  riesgo_laderas  fracturas_count  indice_riesgo_compuesto
0901500011002     015        2           5.0                5.0             1.0            47.92                 0.907312
0901500011093     015        2           5.0                5.0             1.0            47.92                 0.902045
0901500011110     015        2           5.0                5.0             1.0            47.92                 0.898625
0901500011017     015        2           5.0                5.0             1.0            47.92                 0.896122
0901500010818     015        2           5.0                5.0             1.0            47.92                 0.895865
090150001120A     015        2           5.0                5.0             1.0            47.92                 0.894555
0900700010784     007        2           5.0                5.0             1.0            46.00       

## 8. Perfil de cada cluster (v3)

In [33]:
# Perfil detallado de clusters
profile_cols = ['riesgo_sismo', 'riesgo_inundacion', 'riesgo_laderas', 'tipo_suelo',
                'pct_afectacion_sismo', 'severidad_inundacion', 'fracturas_count',
                'ruse_emergencias', 'pob_total', 'imu_2020']
profile_cols = [c for c in profile_cols if c in gdf_v3_km.columns]

cluster_profiles = gdf_v3_km.groupby('cluster')[profile_cols].mean()
cluster_profiles.index = ['Bajo', 'Medio-Bajo', 'Medio-Alto', 'Alto']
print("=== Perfil promedio por cluster (v3) ===")
print(cluster_profiles.round(2).T.to_string())

=== Perfil promedio por cluster (v3) ===
                         Bajo  Medio-Bajo  Medio-Alto     Alto
riesgo_sismo             3.01        3.03        4.55     4.57
riesgo_inundacion        1.10        1.33        4.77     4.87
riesgo_laderas           3.66        1.13        1.04     1.00
tipo_suelo               1.05        1.09        2.83     2.72
pct_afectacion_sismo    56.56       55.82       79.90    78.30
severidad_inundacion    15.02       20.67       96.17    98.01
fracturas_count          1.04        0.45       12.09     0.04
ruse_emergencias        10.07        9.67       17.02    11.18
pob_total             4446.96     3682.69     3980.50  3565.34
imu_2020               121.34      122.01      122.15   122.65


In [34]:
# Heatmap de perfiles
fig, ax = plt.subplots(figsize=(10, 8))
# Normalizar por columna para el heatmap
profiles_norm = cluster_profiles.T.copy()
for col in profiles_norm.columns:
    mn, mx = profiles_norm[col].min(), profiles_norm[col].max()
    if mx > mn:
        profiles_norm[col] = (profiles_norm[col] - mn) / (mx - mn)

sns.heatmap(profiles_norm, annot=cluster_profiles.T.round(2).values, fmt='', 
            cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Perfil de Clusters K-means (v3) - Valores normalizados', fontsize=13, fontweight='bold')
ax.set_xlabel('Cluster')
ax.set_ylabel('Variable')
plt.tight_layout()
plt.savefig(f'{OUT}charts/perfil_clusters_v3.png', bbox_inches='tight')
plt.close()
print("✅ Guardado: perfil_clusters_v3.png")

✅ Guardado: perfil_clusters_v3.png


## 9. Conclusiones

In [35]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║                    CONCLUSIONES DEL MODELADO                       ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  1. COMPARACIÓN DE VERSIONES:                                      ║
║     - v1 tiene menos zonas (1018) y variables, resultando en       ║
║       clusters menos diferenciados                                 ║
║     - v2 y v3 (2453 zonas, 14 features) producen clusters          ║
║       más ricos y diferenciados                                    ║
║     - v3 mejora la separabilidad al corregir pct_afectacion        ║
║                                                                    ║
║  2. ZONAS DE MAYOR RIESGO:                                         ║
║     - Se concentran en alcaldías del oriente de CDMX               ║
║     - Coinciden con suelo tipo III (arenoso/lacustre)              ║
║     - Alto riesgo sísmico + inundación simultáneos                 ║
║     - Presencia significativa de fracturas en estas zonas          ║
║                                                                    ║
║  3. RECOMENDACIONES:                                               ║
║     - Usar v3 como dataset de referencia para análisis futuros     ║
║     - Los 4 clusters capturan bien la gradación de riesgo          ║
║     - Priorizar intervención en zonas cluster "Alto"               ║
║     - Complementar con datos temporales para monitoreo             ║
║                                                                    ║
╚══════════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════════╗
║                    CONCLUSIONES DEL MODELADO                       ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  1. COMPARACIÓN DE VERSIONES:                                      ║
║     - v1 tiene menos zonas (1018) y variables, resultando en       ║
║       clusters menos diferenciados                                 ║
║     - v2 y v3 (2453 zonas, 14 features) producen clusters          ║
║       más ricos y diferenciados                                    ║
║     - v3 mejora la separabilidad al corregir pct_afectacion        ║
║                                                                    ║
║  2. ZONAS DE MAYOR RIESGO:                                         ║
║     - Se concentran en alcaldías del oriente de CDMX               ║
║     - Coinciden con suelo tipo III (arenoso/lacustre)              ║
║